In [ ]:
# ============================================================
# CELL 1: Install Required Libraries
# ============================================================
# groq → official Groq Python client (free LLaMA 3 access)

!pip install groq --quiet

print("Libraries installed!")

Libraries installed!


In [ ]:
# ============================================================
# CELL 2: Import Libraries & Setup API Key
# ============================================================

from groq import Groq
import os

# --- Set your API Key ---
# Option A: Directly (fine for personal use/learning)
API_KEY = "gsk_xxxxxxxxxxxx"   # ← paste your key here

# Option B: Using Colab Secrets (more secure)
# In Colab: click the 🔑 icon on left sidebar → add secret "GROQ_API_KEY"
# from google.colab import userdata
# API_KEY = userdata.get('GROQ_API_KEY')

# Initialize the Groq client
client = Groq(api_key=API_KEY)

print("Groq client initialized!")
print("Model: llama-3.3-70b versatile via Groq")

Groq client initialized!
Model: llama-3.3-70b versatile via Groq


In [ ]:
# ============================================================
# CELL 3: Design the System Prompt (Prompt Engineering)
# ============================================================
# This is the CORE of this task — the system prompt defines:
# 1. Who the chatbot is
# 2. How it responds
# 3. What it refuses to do (safety)
# 4. Tone and style

SYSTEM_PROMPT = """
You are MediBot, a friendly and knowledgeable general health assistant.

YOUR ROLE:
- Answer general health and medical questions clearly and simply
- Explain symptoms, conditions, and medications in plain language
- Provide general wellness and prevention advice
- Help users understand when they should see a doctor

YOUR COMMUNICATION STYLE:
- Use simple, easy-to-understand language (avoid heavy medical jargon)
- Be warm, empathetic, and reassuring
- Keep responses concise but complete (3-5 paragraphs max)
- Use bullet points for lists of symptoms or tips

STRICT SAFETY RULES (you must ALWAYS follow these):
1. NEVER diagnose a specific person with a medical condition
2. NEVER recommend specific prescription drug dosages for self-medication
3. ALWAYS remind users to consult a qualified doctor for personal medical advice
4. If a user describes a medical emergency (chest pain, difficulty breathing,
   severe bleeding), IMMEDIATELY tell them to call emergency services (115 in Pakistan)
5. NEVER provide advice that could replace professional medical consultation
6. If asked about self-harm or suicide, provide crisis helpline information immediately

WHAT YOU CAN DO:
- Explain what a condition generally is
- List common symptoms of a disease
- Explain how a common medication generally works
- Give general wellness tips (diet, exercise, sleep)
- Help users prepare questions for their doctor visit

Remember: You are an educational assistant, not a replacement for a real doctor.
"""

print("System prompt designed!")
print(f"Prompt length: {len(SYSTEM_PROMPT)} characters")
print("\n--- Preview ---")
print(SYSTEM_PROMPT[:300] + "...")

System prompt designed!
Prompt length: 1497 characters

--- Preview ---

You are MediBot, a friendly and knowledgeable general health assistant.

YOUR ROLE:
- Answer general health and medical questions clearly and simply
- Explain symptoms, conditions, and medications in plain language
- Provide general wellness and prevention advice
- Help users understand when they s...


In [ ]:
# ============================================================
# CELL 4: Core Function to Send Query to LLM
# ============================================================

def ask_medibot(user_question, conversation_history=None):
    """
    Send a health question to MediBot and get a response.

    Args:
        user_question (str): The health question from the user
        conversation_history (list): Previous messages for context

    Returns:
        str: MediBot's response
    """

    # Start with system prompt
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    # Add conversation history if provided (for multi-turn chat)
    if conversation_history:
        messages.extend(conversation_history)

    # Add the current user question
    messages.append({
        "role": "user",
        "content": user_question
    })

    # Send to Groq API
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0.7,           # 0=very focused, 1=creative (0.7 is balanced)
        max_tokens=1024,           # max length of response
    )

    # Extract the text response
    return response.choices[0].message.content


print("ask_medibot() function ready!")

ask_medibot() function ready!


In [ ]:
# ============================================================
# CELL 5: Safety Filter — Screen Dangerous Queries
# ============================================================
# Before sending to LLM, check if query is potentially harmful
# This is a rule-based pre-filter (LLM also has safety in system prompt)

def safety_check(user_input):
    """
    Screen user input for dangerous/harmful content.
    Returns (is_safe, warning_message)
    """

    user_input_lower = user_input.lower()

    # Emergency keywords → redirect to emergency services
    emergency_keywords = [
        'chest pain', 'heart attack', 'cant breathe', "can't breathe",
        'difficulty breathing', 'stroke', 'unconscious', 'overdose',
        'severe bleeding', 'not breathing'
    ]

    # Self-harm keywords → redirect to crisis support
    crisis_keywords = [
        'kill myself', 'suicide', 'self harm', 'end my life',
        'want to die', 'hurt myself'
    ]

    # Harmful drug queries
    harmful_keywords = [
        'how to get high', 'drug recipe', 'make drugs',
        'lethal dose', 'poison someone'
    ]

    # Check emergency
    for keyword in emergency_keywords:
        if keyword in user_input_lower:
            return False, (
                "🚨 EMERGENCY ALERT 🚨\n"
                "This sounds like a medical emergency!\n"
                "Please call emergency services IMMEDIATELY:\n"
                "• Pakistan Emergency: 115 (Rescue)\n"
                "• Edhi Foundation: 115\n"
                "• Do NOT wait — call now!"
            )

    # Check crisis
    for keyword in crisis_keywords:
        if keyword in user_input_lower:
            return False, (
                "I'm concerned about what you've shared. "
                "Please reach out for immediate support:\n"
                "• Umang helpline (Pakistan): 0311-7786264\n"
                "• Rozan Counseling: 051-2890505\n"
                "You are not alone. Please talk to someone you trust."
            )

    # Check harmful
    for keyword in harmful_keywords:
        if keyword in user_input_lower:
            return False, (
                "I'm sorry, I can't help with that request. "
                "I'm designed to provide general health education only. "
                "Please consult a healthcare professional for medical guidance."
            )

    return True, None   # safe to proceed


print("Safety filter ready!")

Safety filter ready!


In [ ]:
# ============================================================
# CELL 6: Test MediBot with Sample Health Questions
# ============================================================

def medibot_query(question):
    """Complete pipeline: safety check → LLM → display response"""

    print(f"\n{'='*60}")
    print(f"👤 USER: {question}")
    print(f"{'='*60}")

    # Step 1: Safety check
    is_safe, warning = safety_check(question)

    if not is_safe:
        print(f"🛡️ SAFETY FILTER TRIGGERED:\n{warning}")
        return

    # Step 2: Send to LLM
    print("🤔 MediBot is thinking...")
    response = ask_medibot(question)

    # Step 3: Display response
    print(f"\n🏥 MEDIBOT:\n{response}")
    print(f"\n{'─'*60}")
    print("⚕️  Remember: Always consult a qualified doctor for personal medical advice.")


# --- Test with example queries from the task ---
test_questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "What are the symptoms of diabetes?",
    "How can I improve my sleep quality?",
    "What should I eat to keep my heart healthy?"
]

for question in test_questions:
    medibot_query(question)


👤 USER: What causes a sore throat?
🤔 MediBot is thinking...

🏥 MEDIBOT:
A sore throat can be quite uncomfortable and painful. There are several reasons why you might experience a sore throat. Some common causes include:

* Viral infections, such as the common cold or flu, which can cause inflammation and irritation in the throat
* Bacterial infections, like strep throat, which can be more serious and require antibiotic treatment
* Allergies, which can lead to postnasal drip and throat irritation
* Dry air, which can dry out the throat and make it feel sore
* Shouting or screaming, which can strain the vocal cords and cause throat pain
* Acid reflux, which can cause stomach acid to flow up into the throat and cause irritation

If you're experiencing a sore throat, it's a good idea to try some home remedies to help soothe the discomfort. You can try:

* Staying hydrated by drinking plenty of water
* Gargling with warm salt water
* Using a humidifier to add moisture to the air
* Avoiding

In [ ]:
# ============================================================
# CELL 7: Test Safety Filters Explicitly
# ============================================================

print("Testing safety filters...\n")

dangerous_queries = [
    "I have severe chest pain and can't breathe",
    "I want to kill myself",
    "How to get high on medication"
]

for query in dangerous_queries:
    print(f"\n{'='*60}")
    print(f"👤 USER: {query}")
    print(f"{'='*60}")
    is_safe, warning = safety_check(query)
    if not is_safe:
        print(f"🛡️ SAFETY FILTER:\n{warning}")
    print()

Testing safety filters...


👤 USER: I have severe chest pain and can't breathe
🛡️ SAFETY FILTER:
🚨 EMERGENCY ALERT 🚨
This sounds like a medical emergency!
Please call emergency services IMMEDIATELY:
• Pakistan Emergency: 115 (Rescue)
• Edhi Foundation: 115
• Do NOT wait — call now!


👤 USER: I want to kill myself
🛡️ SAFETY FILTER:
I'm concerned about what you've shared. Please reach out for immediate support:
• Umang helpline (Pakistan): 0311-7786264
• Rozan Counseling: 051-2890505
You are not alone. Please talk to someone you trust.


👤 USER: How to get high on medication
🛡️ SAFETY FILTER:
I'm sorry, I can't help with that request. I'm designed to provide general health education only. Please consult a healthcare professional for medical guidance.



In [ ]:
# ============================================================
# CELL 8: Multi-Turn Conversation with Memory
# ============================================================
# This lets the chatbot remember previous messages in the session
# Each exchange is stored in conversation_history

def run_chat_session(questions):
    """
    Simulate a multi-turn conversation where bot remembers context.

    Args:
        questions (list): List of questions to ask in sequence
    """
    conversation_history = []

    print("🏥 MediBot Chat Session Started")
    print("="*60)

    for question in questions:
        print(f"\n👤 USER: {question}")

        # Safety check
        is_safe, warning = safety_check(question)
        if not is_safe:
            print(f"🛡️ SAFETY FILTER:\n{warning}")
            continue

        # Get response WITH conversation history
        response = ask_medibot(question, conversation_history)
        print(f"\n🏥 MEDIBOT:\n{response}")
        print(f"\n{'─'*50}")

        # Store this exchange in history
        # This is how the bot "remembers" previous messages
        conversation_history.append({"role": "user",      "content": question})
        conversation_history.append({"role": "assistant", "content": response})

    print("\n✅ Chat session ended.")
    print("⚕️  Always consult a real doctor for personal medical advice.")


# --- Test multi-turn conversation ---
# Notice how question 3 references question 1 — bot should remember context!
conversation = [
    "What are the common symptoms of flu?",
    "How is flu different from a common cold?",
    "What home remedies help with the symptoms you mentioned?"  # references earlier answers
]

run_chat_session(conversation)

🏥 MediBot Chat Session Started

👤 USER: What are the common symptoms of flu?

🏥 MEDIBOT:
The flu, also known as influenza, is a common respiratory illness that can affect anyone. If you're wondering if you might have the flu, here are some common symptoms to look out for:

* Sudden onset of fever (usually high)
* Chills
* Cough
* Sore throat
* Runny or stuffy nose
* Headache
* Fatigue (feeling extremely tired)
* Muscle or body aches
* Diarrhea and vomiting (more common in children than adults)

Keep in mind that not everyone with the flu will have all of these symptoms, and the severity of the symptoms can vary from person to person. If you're experiencing any of these symptoms, it's a good idea to rest, stay hydrated, and consult with a doctor if you're concerned. They can provide a proper diagnosis and recommend the best course of treatment.

It's also important to note that if you're experiencing any of the following, you should seek medical attention right away:
* Difficulty breath

In [ ]:
# ============================================================
# CELL 9: Interactive Chat — Type Your Own Questions!
# ============================================================
# This runs a live chat loop in the notebook
# Type 'quit' or 'exit' to stop

def interactive_medibot():
    """
    Live interactive chatbot session.
    Type questions, get real responses.
    Type 'quit' to exit.
    """
    conversation_history = []

    print("🏥 Welcome to MediBot — Your General Health Assistant")
    print("="*60)
    print("Ask me any general health question!")
    print("Type 'quit' or 'exit' to end the session.\n")

    while True:
        # Get user input
        user_input = input("👤 You: ").strip()

        # Exit condition
        if user_input.lower() in ['quit', 'exit', 'bye', 'q']:
            print("\n🏥 MediBot: Take care and stay healthy! Goodbye! 👋")
            print("⚕️  Remember to consult a real doctor for personal advice.")
            break

        # Skip empty input
        if not user_input:
            continue

        # Safety check
        is_safe, warning = safety_check(user_input)
        if not is_safe:
            print(f"\n🛡️ {warning}\n")
            continue

        # Get and display response
        print("\n🤔 Thinking...")
        response = ask_medibot(user_input, conversation_history)
        print(f"\n🏥 MediBot: {response}\n")
        print("─"*50)

        # Update history
        conversation_history.append({"role": "user",      "content": user_input})
        conversation_history.append({"role": "assistant", "content": response})


# Run the interactive chatbot
interactive_medibot()

🏥 Welcome to MediBot — Your General Health Assistant
Ask me any general health question!
Type 'quit' or 'exit' to end the session.

👤 You: im having some fever..please help me out

🤔 Thinking...

🏥 MediBot: I'm so sorry to hear that you're not feeling well. A fever can be uncomfortable and worrying. Don't worry, I'm here to help you understand what might be going on and what you can do to feel better.

First, let's talk about what a fever is. A fever is a temporary increase in your body temperature, usually above 98.6°F (37°C). It's a common symptom of many illnesses, including infections, viruses, and flu. Some common symptoms of a fever include:
* Elevated body temperature
* Chills or sweating
* Headache or body aches
* Fatigue or weakness
* Loss of appetite

To help manage your fever, you can try some simple things at home:
* Drink plenty of fluids, such as water, tea, or soup, to stay hydrated
* Rest and avoid strenuous activities
* Take over-the-counter medication, such as acetami

In [ ]:
# ============================================================
# CELL 10: Prompt Engineering — See How Prompts Change Responses
# ============================================================
# This demonstrates WHY prompt engineering matters
# Same question, 3 different system prompts → 3 different responses

question = "What causes headaches?"

prompts = {
    "No System Prompt (Raw LLM)": None,

    "MediBot (Our Designed Prompt)": SYSTEM_PROMPT,

    "Overly Cautious Prompt": """
        You are an extremely cautious medical assistant.
        For every question, primarily tell users to see a doctor.
        Keep all answers under 2 sentences.
        Always end with: 'Please consult your physician immediately.'
    """,
}

print(f"Question: '{question}'")
print("="*60)

for prompt_name, system_prompt in prompts.items():
    print(f"\n📋 PROMPT: {prompt_name}")
    print("─"*40)

    if system_prompt is None:
        messages = [{"role": "user", "content": question}]
    else:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question}
        ]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0.7,
        max_tokens=300
    )

    print(response.choices[0].message.content)
    print()

# WHAT YOU SHOULD SEE:
# Raw LLM → generic answer, no warmth, no safety disclaimer
# MediBot  → warm, structured, ends with doctor recommendation
# Cautious → very short, mostly just "see a doctor"
# This proves: prompt design DIRECTLY controls LLM behavior

Question: 'What causes headaches?'

📋 PROMPT: No System Prompt (Raw LLM)
────────────────────────────────────────
Headaches are a common and often debilitating condition that can be caused by a wide range of factors. Here are some of the most common causes of headaches:

1. **Tension and Stress**: Tight muscles in the neck and scalp can lead to tension headaches, which are the most common type of headache.
2. **Migraines**: A neurological disorder that can cause severe, throbbing headaches, often accompanied by sensitivity to light, sound, and nausea.
3. **Dehydration**: Not drinking enough water or losing fluids can lead to dehydration, which can cause headaches.
4. **Poor Posture**: Poor posture can lead to muscle strain in the neck and back, which can cause headaches.
5. **Sinus Pressure**: Sinus infections, allergies, or colds can cause headaches due to pressure and congestion in the sinuses.
6. **Hormonal Changes**: Hormonal fluctuations, such as those experienced during menstruat

In [ ]:
readme_content = """# Task 4: General Health Query Chatbot (Prompt Engineering)

## Objective
Build a chatbot that answers general health-related questions
using a Large Language Model (LLM) with carefully engineered prompts
and safety filters.

## Tools & APIs
- **LLM:** LLaMA 3.3 70B via Groq API (free)
- **Library:** `groq` Python client
- **Interface:** Jupyter Notebook (interactive chat loop)

## Dataset
No training dataset used. This task is prompt engineering based —
the LLM is already trained. We engineer the system prompt to
shape its behavior.

## Models Applied
- **LLaMA 3.3-70b-versatile** via Groq API
- Rule-based safety filter (pre-LLM screening layer)

## Prompt Engineering Design
System prompt defined:
- Bot identity: MediBot — friendly health assistant
- Communication style: warm, simple, empathetic
- Hard safety rules: no diagnosis, no prescription dosages,
  always recommend consulting a doctor
- Emergency handling: redirect to emergency services (115 Pakistan)
- Crisis handling: redirect to mental health helplines

## Safety Architecture (Two Layers)
1. **Rule-based filter** — keyword matching for emergencies,
   crisis situations, and harmful drug queries
2. **LLM system prompt** — handles nuanced safety in responses

## Key Results & Findings
- Prompt design directly controls LLM tone and behavior
- Two-layer safety is more robust than relying on LLM alone
- Multi-turn memory (conversation history list) makes chat feel natural
- Same LLM gave completely different responses with different prompts
- Temperature=0.7 gave balanced, natural responses

## Example Queries Tested
- "What causes a sore throat?"
- "Is paracetamol safe for children?"
- "What are symptoms of diabetes?"
- "How can I improve my sleep quality?"

## Conclusion
Prompt engineering is a powerful technique for customizing LLM behavior
without any model training. A well-designed system prompt with
layered safety filters can create a reliable, domain-specific chatbot.
"""

with open("README.md", "w") as f:
    f.write(readme_content)

print("README.md created successfully!")

README.md created successfully!


## Task 4 — Key Insights: Health Query Chatbot

**Objective:** Build a prompt-engineered health chatbot using a free LLM

**Tools Used:**
- Groq API (free) with LLaMA 3.3-70b Versatile model
- Python groq library for API calls
- Rule-based safety filter (pre-LLM screening)

**Prompt Engineering Decisions:**
- System prompt defines role, tone, capabilities, and hard limits
- Temperature = 0.7 for balanced, natural responses
- Multi-turn memory via conversation_history list

**Safety Architecture (Two Layers):**
1. Rule-based filter → catches emergency/crisis keywords instantly
2. LLM system prompt → handles nuanced safety within responses

**Key Findings:**
- Prompt design directly controls LLM behavior and tone
- Two-layer safety is more robust than relying on LLM alone
- Multi-turn memory makes conversations feel natural
- Same LLM gives completely different responses with different prompts

**Limitations:**
- Rule-based filter can miss paraphrased dangerous queries
- LLM may occasionally hallucinate medical information
- Not suitable for actual medical deployment without clinical review